# Cairnly Assistant – 2. Run the chatbot

Runs the chatbot evaluated in `1_model_evaluation.ipynb`: `openai/gpt-oss-120b` on Groq, e5 vector search (top 3), few-shot prompt, order tool and guardrails. The chatbot code is in `chatbot.py`; all explanations are in `chatbot_documentation.docx`.

**Run:** keep next to `chatbot.py` and `data/`, *Run all*, paste a Groq API key, then chat in Section 2 or in the browser (Section 3).

## 1. Setup

Importing `chatbot` loads the e5 model (~470 MB download on the first run). The API key is read with `getpass()` and never stored.

In [ ]:
%pip install -q groq numpy sentence-transformers streamlit

In [1]:
from getpass import getpass

import chatbot                                   # the chatbot itself (chatbot.py)
from chatbot import ChatSession, show

chatbot.connect(getpass("Paste your Groq API key (input is hidden): "))
print("Groq client created.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Groq client created.


## 2. Chat with the bot

Type a message; an empty line or `quit` stops. Try: *Where is my order A1001?*, then switch language, e.g. *Mikor érkezik meg?*

In [ ]:
chat = ChatSession()
while True:
    q = input("You: ").strip()
    if q.lower() in {"quit", "exit", ""}:
        break
    show(chat.ask(q))

Customer : szia szeretném tudni a rendelésem állapotát
Bot      : Szia! Ahhoz, hogy meg tudjam nézni a rendelésed állapotát, kérlek írd meg a rendelésed számát (pl.
           A1234).
           [Orders & Delivery | urgency medium | sources=['KB-03'] | tools=[] | blocked=None]

Customer : so the code for my order is A1001
Bot      : A1001 számú rendelésed már feladva van, a szállító: SwiftParcel, nyomkövetési kód: SP-552190. A
           becsült kézbesítés 2026‑10‑12.
           [Orders & Delivery | urgency low | sources=['KB-03'] | tools=[('get_order_status', '{"order_id":"A1001"}', 'ok')] | blocked=None]

Customer : köszi
Bot      : Szívesen! Ha bármiben segíthetek még, csak szólj.
           [Out of scope | urgency low | sources=[] | tools=[] | blocked=None]



## 3. Web chat (Streamlit)

Running the cell builds the Streamlit page from `web_chat()` and starts it right away with the API key entered in Section 1. Open http://localhost:8501; reload the page for a new conversation. The page loads the chatbot only once, with Streamlit's `@st.cache_resource` ([caching docs](https://docs.streamlit.io/develop/concepts/architecture/caching)), so only the first answer is slow. Interrupt the cell (■) to stop it.

In [ ]:
import inspect, os, subprocess, sys, tempfile, time


def web_chat():
    '''The Streamlit page. It runs in its own process: one conversation per browser tab, reload for a new one.'''
    import os
    import streamlit as st

    @st.cache_resource
    def load_chatbot():
        '''Runs once per server; every rerun and browser tab reuses the result (e5 model, embeddings, Groq client).'''
        import chatbot
        chatbot.connect(os.environ["GROQ_API_KEY"])
        return chatbot

    st.set_page_config(page_title="Cairnly Assistant", page_icon="⛺")
    st.title("Cairnly Assistant")
    chatbot = load_chatbot()
    if "chat" not in st.session_state:
        st.session_state.chat = chatbot.ChatSession()
        st.session_state.messages = []
    for role, text in st.session_state.messages:
        st.chat_message(role).markdown(text)
    if question := st.chat_input("Ask about products, orders or policies"):
        st.chat_message("user").markdown(question)
        with st.spinner("Thinking..."):
            r = st.session_state.chat.ask(question)
        answer = r["answer"] if r["category"] else "Sorry, something went wrong. Please try again."   # no readable JSON reply
        st.chat_message("assistant").markdown(answer)
        st.session_state.messages += [("user", question), ("assistant", answer)]


# Build: Streamlit runs a page from a script, so web_chat() is written to the system temp folder (not into the project)
page = os.path.join(tempfile.gettempdir(), "cairnly_web_chat.py")
with open(page, "w", encoding="utf-8") as f:
    f.write(inspect.getsource(web_chat) + "\n\nweb_chat()\n")

# Run: own process, only on this computer; the API key from Section 1 is passed in memory (environment), never saved
server = subprocess.Popen([sys.executable, "-m", "streamlit", "run", page, "--server.address", "localhost",
                           "--server.port", "8501", "--server.headless", "true", "--server.fileWatcherType", "none"],
                          env={**os.environ, "GROQ_API_KEY": chatbot.client.api_key})
print("Web chat: http://localhost:8501 (the first answer takes a few seconds). Interrupt this cell (■) to stop it.")
try:
    while server.poll() is None:
        time.sleep(0.5)
    print("The web chat stopped on its own: is port 8501 already used by another Streamlit app?")
except KeyboardInterrupt:
    print("Web chat stopped.")
finally:
    server.terminate()